# Solutions · Chapter 03-03 · Error bars from the data you actually have

E12 and E16 are the two to attempt before reading. E12's failure is worse than the exercise text
suggests, and E16 has a clean solution that beats the bootstrap's 0.000 coverage outright.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pop_rng = np.random.default_rng(11)
population = pop_rng.gamma(2.0, 3.0, 1_000_000)
TRUE_MEAN = population.mean()
TRUE_MEDIAN = float(np.median(population))

week_rng = np.random.default_rng(1)
week = population[week_rng.integers(0, len(population), 7)]

print("week:", np.sort(week).round(2), " mean %.3f" % week.mean())
print("population mean %.4f  median %.4f" % (TRUE_MEAN, TRUE_MEDIAN))

## E1 · What it resamples, and why that is reasonable

It resamples **your own data, with replacement, at the same sample size**, recomputing the statistic
each time.

It is reasonable because the sampling distribution describes what happens when you draw `n`
observations from the population, and your sample is the best available estimate of that population -
so drawing `n` observations from your sample imitates the same process. The imitation is good exactly
where your sample resembles the population, which is the middle, and poor at the edges.

## E2 · Centred on your estimate, not the truth

The resamples are built from your values, so their average is your sample's average. The bootstrap
never sees the population and has no way to know your sample sat low or high.

**What it therefore cannot detect: bias.** It measures how much an estimate would *wobble*, not how
far it sits from the answer. A sample from a broken sensor, an unrepresentative panel, or a survey
with the censoring from 02-03 will produce a confident, narrow, wrong interval - and everything that
makes it wrong is invisible from inside the data. **Uncertainty is not the same as error**, and only
one of the two is computable.

## E3 · What 95% confidence means

> "If I repeated the whole procedure - draw a sample of this size, compute this interval - many
> times, 95% of the intervals produced would contain the true value. This particular interval either
> contains it or does not, and I cannot tell which."

The trap being avoided is assigning a probability to the *true value*, which is fixed. The probability
belongs to the interval-generating procedure.

## E4 · Bootstrap resamples by hand

In [ ]:
small = np.array([2, 4, 4, 9, 11])

examples = [
    [2, 4, 4, 9, 11],     # the original, which is one possible resample
    [4, 4, 4, 9, 9],
    [2, 2, 2, 4, 4],
]
for resample in examples:
    print("%-22s mean %.1f" % (str(resample), np.mean(resample)))

print()
print("largest possible mean : %.1f  (all 11s)" % small.max())
print("smallest possible mean: %.1f  (all 2s)" % small.min())

The extreme cases are resamples that pick the same value five times, so **the largest possible
bootstrap mean is 11 and the smallest is 2** - the range of the original data.

This is the maximum failure in miniature: **every bootstrap statistic is trapped inside the range of
the observed values.** For a mean that hardly matters, since means near the edges are
astronomically unlikely. For a maximum it is fatal, because the maximum sits exactly on the boundary.

## E5 · Interval positions

In [ ]:
for level in [0.90, 0.95, 0.99]:
    in_each_tail = round(10_000 * (1 - level) / 2)
    print("%.0f%% interval: positions %5d and %5d of 10,000 sorted resamples  (%d values in each tail)"
          % (100 * level, in_each_tail, 10_000 - in_each_tail, in_each_tail))

90%: positions 500 and 9,500. 99%: positions 50 and 9,950.

**The 99% interval is estimated least reliably.** Its ends are determined by only 50 resamples in
each tail, so they jump around from one bootstrap run to the next; the 90% interval's ends rest on
500 values each and are far steadier.

The general rule: **the further into the tail an interval reaches, the more resamples it needs.** A
99% interval wants 50,000 or more repeats to be stable, and a 99.9% interval is not something the
percentile bootstrap should be asked for at all - which is the same edge problem as the maximum,
approached gradually.

## E6 · Intervals for three statistics

In [ ]:
def interval(sample, statistic, level=0.95, repeats=10_000, seed=0):
    rng = np.random.default_rng(seed)
    resamples = sample[rng.integers(0, len(sample), (repeats, len(sample)))]
    values = statistic(resamples, axis=1)
    tail = (1 - level) / 2
    lo, hi = np.quantile(values, [tail, 1 - tail])
    return float(statistic(sample[None, :], axis=1)[0]), float(lo), float(hi)


rows = []
for name, statistic in [("mean", np.mean),
                        ("median", np.median),
                        ("90th percentile", lambda a, axis: np.quantile(a, 0.9, axis=axis))]:
    estimate, lo, hi = interval(week, statistic)
    rows.append({"statistic": name, "estimate": round(estimate, 3),
                 "low": round(lo, 3), "high": round(hi, 3), "width": round(hi - lo, 3)})
print(pd.DataFrame(rows).to_string(index=False))

**The 90th percentile is widest by a long way - 10.878 against 6.250 for the mean.**

Two reasons, and both are the chapter's theme:

- The 90th percentile of seven values is determined by the largest one or two, so it inherits all
  their variability. The mean averages seven values and dilutes it.
- It is close to the edge, where the bootstrap is weakest. The interval runs from 3.54 to 14.42, and
  14.42 is exactly the largest value in the sample - **the upper end is pinned to the sample maximum
  and cannot move past it**. That is the maximum's failure, appearing in a milder form.

The practical reading: asking for a 90th percentile from seven observations is asking a question the
data cannot answer, and here the interval is honest enough to say so by being enormous.

## E7 · Coverage for the median

In [ ]:
cover_rng = np.random.default_rng(0)

rows = []
for n in [7, 31, 101]:
    hits = 0
    for _ in range(1500):
        sample = population[cover_rng.integers(0, len(population), n)]
        values = np.median(sample[cover_rng.integers(0, n, (1000, n))], axis=1)
        lo, hi = np.quantile(values, [0.025, 0.975])
        hits += lo <= TRUE_MEDIAN <= hi
    rows.append({"n": n, "median coverage": round(hits / 1500, 3)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nfor comparison, the chapter's mean coverage was 0.859 at n=7, 0.933 at n=30, 0.939 at n=100")

**The median does slightly better: 0.889 at n = 7 against the mean's 0.859**, and 0.939 against 0.933
at around 30.

The reason is 03-02's last section. The bootstrap's small-sample problem is that resamples
under-represent the tail; the median barely notices the tail, so it has less to under-represent. The
mean's interval depends on how much extreme values vary, which is exactly what seven observations
estimate worst.

Both are still short of 0.95 at n = 7, so this is a mitigation and not a fix.

## E8 · The basic interval instead of the percentile interval

In [ ]:
compare_rng = np.random.default_rng(5)

rows = []
for n in [7, 30]:
    percentile_hits, basic_hits = 0, 0
    for _ in range(1500):
        sample = population[compare_rng.integers(0, len(population), n)]
        values = sample[compare_rng.integers(0, n, (1000, n))].mean(axis=1)
        lo, hi = np.quantile(values, [0.025, 0.975])
        percentile_hits += lo <= TRUE_MEAN <= hi
        basic_low, basic_high = 2 * sample.mean() - hi, 2 * sample.mean() - lo
        basic_hits += basic_low <= TRUE_MEAN <= basic_high
    rows.append({"n": n, "percentile coverage": round(percentile_hits / 1500, 3),
                 "basic coverage": round(basic_hits / 1500, 3)})
print(pd.DataFrame(rows).to_string(index=False))

**The percentile interval wins here: 0.856 against 0.842 at n = 7, and 0.939 against 0.916 at n = 30.**

The two differ only when the bootstrap distribution is asymmetric. The basic interval reflects the
resamples through the estimate, on the theory that the *error* is what should be reflected; the
percentile interval keeps them where they fall.

This population is right-skewed, so the bootstrap distribution leans right, and reflecting it moves
the interval in the wrong direction. On a symmetric population the two would be nearly identical.

**The honest conclusion is not "percentile is better".** It is that interval methods have different
failure modes and the ranking depends on the data - which is why BCa exists, why coverage
simulations like this one are the way to settle such arguments, and why anyone reporting an interval
on a small skewed sample should say which method produced it.

## E9 · Bootstrapping a correlation

In [ ]:
rows = []
for n in [40, 200]:
    gen = np.random.default_rng(7)
    x = gen.normal(0, 1, n)
    y = 0.5 * x + gen.normal(0, np.sqrt(1 - 0.25), n)      # true correlation 0.5
    observed = float(np.corrcoef(x, y)[0, 1])

    boot_rng = np.random.default_rng(8)
    values = np.empty(5000)
    for i in range(5000):
        idx = boot_rng.integers(0, n, n)                    # resample PAIRS, not columns
        values[i] = np.corrcoef(x[idx], y[idx])[0, 1]
    lo, hi = np.quantile(values, [0.025, 0.975])
    rows.append({"n": n, "observed r": round(observed, 3), "low": round(lo, 3), "high": round(hi, 3),
                 "width": round(hi - lo, 3), "below estimate": round(observed - lo, 3),
                 "above estimate": round(hi - observed, 3)})
print(pd.DataFrame(rows).to_string(index=False))

**The width falls from 0.465 to 0.212** - a factor of 2.2, against `sqrt(200/40) = 2.24`. The
`1/sqrt(n)` law holds for correlations too.

**The intervals are not symmetric**: at n = 40 the interval reaches 0.254 below the estimate and
0.211 above. Correlations are bounded at 1, so the sampling distribution is squashed on the upper
side, and the further the true value is from zero the more pronounced this becomes. Any report of the
form "r = 0.37 plus or minus 0.23" is imposing a symmetry that is not there.

**The detail that matters most in the code: resample the pairs.** `x[idx], y[idx]` uses the same
indices for both, keeping each observation intact. Resampling the two columns independently would
destroy the pairing and drive every bootstrap correlation towards zero - which is the same mistake
E12 is about.

## E10 · "Improved by 0.4%, CI -0.9% to 1.7%"

**What is wrong with the conclusion:** the interval includes 1.7%, so the data is entirely consistent
with a worthwhile improvement. "Not proven to work" has been read as "shown not to work", and those
are different findings. An interval spanning from -0.9 to 1.7 says the experiment was **too small to
answer the question**, not that the answer is no.

**What to say instead:** "The result is inconclusive. The data is compatible with anything from a
0.9% harm to a 1.7% gain, so we cannot act on it either way. If a 1% improvement is worth having, we
need roughly *(current width / target width)^2* times the sample - about four times as many users -
and I would rather run it for another three weeks than call it dead."

That reply also protects against the more expensive error: quietly shelving features that work,
which leaves no evidence behind and is never reviewed.

## E11 · Two models, overlapping intervals

**Why overlap is not sufficient:** the two scores come from **the same 500 test rows**, so their
errors are strongly correlated - the rows that are hard for one model are usually hard for the other.
Two separate intervals treat the models as independently measured and throw that away. Overlapping
intervals are a conservative test: they can overlap substantially while the difference is consistent
and clear.

**What to do instead: bootstrap the difference, resampling test rows rather than models.** Resample
the 500 row indices once per iteration, score *both* models on that same resample, take the
difference, and build the interval on the differences. The shared rows cancel, which is exactly the
variation you wanted to remove.

This is a paired comparison, it is nearly always more powerful than comparing two intervals, and it
is what module 07 uses to compare models properly.

## E12 · Bootstrapping a time series by resampling days

In [ ]:
series_rng = np.random.default_rng(13)
days = 200
trend = np.arange(days) * 0.05                      # the true trend: 0.05 per day
noise = np.zeros(days)
for i in range(1, days):
    noise[i] = 0.85 * noise[i - 1] + series_rng.normal(0, 1)   # strongly autocorrelated
series = 20 + trend + noise


def slope(values):
    return np.polyfit(np.arange(len(values)), values, 1)[0]


print("observed slope: %.4f per day   (the true trend is 0.0500)" % slope(series))

naive_rng = np.random.default_rng(14)
naive = np.array([slope(series[naive_rng.integers(0, days, days)]) for _ in range(2000)])

truth_rng = np.random.default_rng(15)
honest = []
for _ in range(2000):
    fresh = np.zeros(days)
    for i in range(1, days):
        fresh[i] = 0.85 * fresh[i - 1] + truth_rng.normal(0, 1)
    honest.append(slope(20 + trend + fresh))
honest = np.array(honest)

print()
print("naive bootstrap (resampling days) : sd %.5f   95%% interval %.4f to %.4f"
      % (naive.std(), *np.quantile(naive, [0.025, 0.975])))
print("the truth (regenerating the series): sd %.5f   95%% interval %.4f to %.4f"
      % (honest.std(), *np.quantile(honest, [0.025, 0.975])))

### It is worse than too narrow

The naive interval is **1.8 times too narrow**, which is what the exercise asked about. But look at
where it sits: **-0.0081 to 0.0077**, centred on **zero**, while the observed slope is 0.0482 and the
truth is 0.05.

**The interval does not contain the estimate it is supposed to describe.**

The cause: resampling days with replacement discards the order. A resample is 200 days in random
sequence, and the slope of a randomly ordered series is zero by construction. The bootstrap is
faithfully describing the variability of a quantity that no longer means anything.

**The direction of the error, in general:** ordinary resampling destroys dependence, and dependent
observations carry less information than independent ones, so intervals come out **too narrow** -
often dramatically, and always in the direction that makes you overconfident. Autocorrelated data is
the most common case, and clustered data is the next: 200 measurements from 20 patients are not 200
independent observations, and treating them as such can shrink an interval by a factor of three.

**The fix: the block bootstrap.** Resample contiguous *blocks* - say 20 chunks of 10 consecutive
days - so the dependence inside each block survives. The block length has to be long enough to
capture the dependence and short enough to leave blocks to shuffle, and choosing it is a real
judgement call. Module 09 does this properly, along with why a time series needs a different
train-test split as well.

## E13 · "How confident are you in 91% accuracy?"

> "It depends almost entirely on the test set, so my first question is how many rows it has. With a
> thousand rows the standard error of an accuracy of 0.91 is `sqrt(0.91 x 0.09 / 1000)`, about **0.9
> percentage points**, so the honest statement is 91% plus or minus roughly two points. With a
> hundred rows it would be plus or minus six, and the number would not support a decision. I would
> compute that interval by bootstrapping the test predictions rather than by formula, so the same
> method covers precision and recall too. And I would ask whether the test set was drawn the same way
> as the data the model will actually see - because if it was not, the interval describes the
> precision of a number that is measuring the wrong thing."

Five sentences, one computed number, one question about the data, and it closes on the distinction
from E2: an interval bounds wobble, not wrongness.

## E14 · Median survival from 60 patients

**How to put an interval on it:** bootstrap the patients. Resample 60 patients with replacement,
recompute the median survival difference between arms, repeat ten thousand times, take the 2.5th and
97.5th percentiles. It works for a median where no simple formula does, and it handles the
skewed survival distribution without assuming a shape.

**What could make it unreliable here:**

- **Censoring.** Patients still alive at the end of the study have no survival time - the exact
  situation from 02-03. The median is only computable at all if more than half have an event, and
  naive bootstrapping of censored data biases the result. Survival analysis exists for this reason.
- **n = 60 split across two arms** is 30 each, and the coverage table says intervals at that size are
  optimistic.
- **Dependence** if patients are clustered by hospital or by treating clinician.

**What I would check first: how many patients are censored, and whether censoring differs between the
arms.** If the better arm has more patients still alive, the median understates its benefit, and no
interval computed from the observed times can repair that.

## E15 · For the manager

> "We measured one week and got an average. If we had watched a different week we would have got a
> slightly different number, and the question is how different. We cannot go back and watch fifty
> more weeks, so we do the next best thing: take the seven days we did watch, and build thousands of
> imaginary weeks by drawing days from them at random, allowing repeats. Each imaginary week gives its
> own average. The range those averages fall into tells us how much our real answer could have moved
> by luck - and that range is what we should quote, rather than a single number that looks more exact
> than it is."

87 words, no jargon, and it ends on why the technique is worth the trouble.

## E16 · An interval for the maximum that actually works

The bootstrap fails because a resample cannot exceed the sample maximum. So use a method that
**scales the sample maximum upwards** rather than resampling it.

For values spread uniformly between 0 and some unknown upper bound `theta`, the largest of `n`
observations is on average `n / (n + 1)` of the way to `theta` - it is always too small, by a factor
that shrinks predictably with `n`. That relationship is exact, and it can be inverted into an
interval: the ratio `sample max / theta` behaves in a known way, so dividing the sample maximum by
that ratio's percentiles bounds `theta`.

In [ ]:
bound_rng = np.random.default_rng(4)
bounded = bound_rng.uniform(0, 10, 1_000_000)
TRUE_MAX = 10.0

check_rng = np.random.default_rng(9)
rows = []
for n in [30, 200, 1000]:
    hits, widths = 0, []
    for _ in range(2000):
        sample = bounded[check_rng.integers(0, len(bounded), n)]
        observed_max = sample.max()
        lo = observed_max / 0.975 ** (1 / n)     # invert the known ratio
        hi = observed_max / 0.025 ** (1 / n)
        widths.append(hi - lo)
        hits += lo <= TRUE_MAX <= hi
    rows.append({"n": n, "coverage": round(hits / 2000, 3),
                 "average width": round(float(np.mean(widths)), 3),
                 "bootstrap coverage": 0.0})
print(pd.DataFrame(rows).to_string(index=False))

**Coverage 0.955, 0.951, 0.952 - against the bootstrap's 0.000.** And unlike the bootstrap's interval,
this one lies mostly *above* the sample maximum, which is where the answer actually is.

The lesson is the one the chapter has been building towards. The bootstrap is general because it
assumes almost nothing, and that is the same reason it fails here: it has no way to reason about
values it has never seen. The method above works because it uses something the bootstrap does not
have - **the knowledge that the distribution is uniform** - to extrapolate beyond the data.

That is the permanent trade in statistics: **assumptions buy you extrapolation, and cost you
whatever they get wrong.** If the population were not uniform, this interval would be confidently
incorrect in a way the bootstrap never is. The bootstrap's failure on maxima is loud and total; a
wrong parametric assumption fails quietly. Neither is safer in general, and the reason to know both
is so that the choice is a decision rather than a habit.

## Where to go next

**03-04 · Probability and conditional probability, with counts.** Module 03 turns from summarising and
its uncertainty to reasoning about uncertainty directly - starting, deliberately, with tables you can
count rather than with axioms.